# LangChain 静态检索：VectorStore → Retriever → Tool

这份 notebook 只解释 LangChain 的接口边界，不建设持久化索引系统：

| 层 | 当前示例 | 职责 |
| --- | --- | --- |
| VectorStore | InMemoryVectorStore | 保存向量并执行搜索，可返回 score |
| Retriever | as_retriever / BM25Retriever | 接收字符串并返回 Document，是 Runnable |
| Tool | create_retriever_tool | 给 Agent 暴露已验证的 Retriever 接口 |

数据量只有几十个 chunk，因此统一使用 InMemoryVectorStore。LangChain 负责接口和编排，
不会自动改善 embedding、中文分词或底层召回质量。

## 1. 最小、自包含的数据准备

本文件直接读取 v4 TXT、切分并建立内存向量库，不导入项目内检索模块，
也不依赖其他 notebook 的变量。

In [1]:
from __future__ import annotations

import hashlib
import re
import unicodedata
import warnings
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.tools import create_retriever_tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 当前安装版导入 BM25Retriever 时会给出 community 迁移提示；
# 本 notebook 只记录当前可运行接口，不把警告刷进教学输出。
warnings.filterwarnings(
    "ignore",
    message=".*langchain-community.*being sunset.*",
    category=DeprecationWarning,
)
from langchain_community.retrievers.bm25 import BM25Retriever


repo_root = Path.cwd().resolve()
if repo_root.name == "ZZworkbench":
    repo_root = repo_root.parent

text_dir = repo_root / "knowledge" / "project_progress" / "texts" / "v4"
embed_path = (
    Path("/mnt/e/local_models/embedding")
    / "iic--nlp_gte_sentence-embedding_chinese-base"
)
assert text_dir.is_dir()
assert embed_path.is_dir()
print({"repo_ok": repo_root.name == "pipelines_rag", "model": embed_path.name})

{'repo_ok': True, 'model': 'iic--nlp_gte_sentence-embedding_chinese-base'}


In [2]:
documents: list[Document] = []
for path in sorted(text_dir.glob("*.txt")):
    text = path.read_text(encoding="utf-8-sig").strip()
    relative_source = path.relative_to(repo_root).as_posix()
    document_id = "doc-" + hashlib.sha1(
        f"{relative_source}\n{text}".encode("utf-8")
    ).hexdigest()[:12]
    documents.append(
        Document(
            id=document_id,
            page_content=text,
            metadata={
                "source": relative_source,
                "source_name": path.name,
                "document_id": document_id,
                "version": path.parent.name,
            },
        )
    )

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", "！", "？", "；", "，", " ", ""],
    chunk_size=640,
    chunk_overlap=128,
    add_start_index=True,
)
chunks: list[Document] = []
for document in documents:
    for index, chunk in enumerate(splitter.split_documents([document])):
        chunk_id = f"{document.id}:chunk-{index:03d}"
        chunks.append(
            Document(
                id=chunk_id,
                page_content=chunk.page_content,
                metadata={**chunk.metadata, "chunk_id": chunk_id},
            )
        )

embeddings = HuggingFaceEmbeddings(
    model=str(embed_path),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
    query_encode_kwargs={"normalize_embeddings": True},
    show_progress=False,
)
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(
    documents=chunks,
    ids=[str(chunk.id) for chunk in chunks],
)
print({"documents": len(documents), "chunks": len(chunks)})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'documents': 8, 'chunks': 88}


## 2. VectorStore：搜索与 score

VectorStore 不是 Runnable；它提供 add/delete/get/search 等存储接口。
similarity_search 只返回 Document，similarity_search_with_score 同时保留底层分数。

当前 InMemoryVectorStore 返回 cosine similarity，越大越相关。换 VectorStore 后必须重新
检查 score 是相似度还是距离。

In [3]:
query = "珠海黄金输变电工程的主体结构封顶计划什么时候完成？"
scored = vector_store.similarity_search_with_score(query, k=4)

for rank, (doc, score) in enumerate(scored, start=1):
    print(
        {
            "rank": rank,
            "source": doc.metadata["source_name"],
            "chunk_id": doc.metadata["chunk_id"],
            "cosine": round(float(score), 4),
        }
    )

{'rank': 1, 'source': '110kV黄金输变电工程三级进度计划.txt', 'chunk_id': 'doc-b94ad48bc283:chunk-000', 'cosine': 0.8058}
{'rank': 2, 'source': '三级进度计划-土建.txt', 'chunk_id': 'doc-51458ce8b85c:chunk-000', 'cosine': 0.7942}
{'rank': 3, 'source': '珠海110千伏江湾输变电工程施工进度计划（202.txt', 'chunk_id': 'doc-e82a731a86c8:chunk-000', 'cosine': 0.7894}
{'rank': 4, 'source': '三虎输变电工程三级进度计划土建部分.txt', 'chunk_id': 'doc-0881e34d986f:chunk-000', 'cosine': 0.7863}


## 3. VectorStoreRetriever：similarity 与 MMR

as_retriever 返回 BaseRetriever。Retriever 是 Runnable，因此使用 invoke、batch、
ainvoke 等统一接口，但默认只返回 Document，不保留原始 score。

search_kwargs 由底层 VectorStore 解释：

- similarity：k、filter；
- mmr：k、fetch_k、lambda_mult、filter；
- fetch_k 应明显大于 k，否则 MMR 没有足够候选做多样性选择。

In [4]:
target_source = "110kV黄金输变电工程三级进度计划.txt"
metadata_filter = lambda doc: doc.metadata["source_name"] == target_source

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4, "filter": metadata_filter},
)
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 16,
        "lambda_mult": 0.65,
        "filter": metadata_filter,
    },
)

for name, retriever in [
    ("similarity", similarity_retriever),
    ("mmr", mmr_retriever),
]:
    result = retriever.invoke(query)
    print(name, [doc.metadata["chunk_id"] for doc in result])

similarity ['doc-b94ad48bc283:chunk-000', 'doc-b94ad48bc283:chunk-005', 'doc-b94ad48bc283:chunk-002', 'doc-b94ad48bc283:chunk-001']
mmr ['doc-b94ad48bc283:chunk-000', 'doc-b94ad48bc283:chunk-005', 'doc-b94ad48bc283:chunk-002', 'doc-b94ad48bc283:chunk-001']


## 4. Runnable：同一个 Retriever 可以 batch

Runnable 是调用协议，不是新的检索算法。batch 只是统一处理多个输入；
每个问题仍由同一个 similarity retriever 独立检索。

In [5]:
batch_queries = [
    "主体结构封顶什么时候完成？",
    "变电站电气包括哪些任务？",
]
batch_results = similarity_retriever.batch(batch_queries)
print(
    [
        {
            "query": current_query,
            "count": len(result),
            "sources": sorted(
                {doc.metadata["source_name"] for doc in result}
            ),
        }
        for current_query, result in zip(batch_queries, batch_results, strict=True)
    ]
)

[{'query': '主体结构封顶什么时候完成？', 'count': 4, 'sources': ['110kV黄金输变电工程三级进度计划.txt']}, {'query': '变电站电气包括哪些任务？', 'count': 4, 'sources': ['110kV黄金输变电工程三级进度计划.txt']}]


## 5. BM25Retriever 也是 Retriever

BM25Retriever 的默认 preprocess_func 使用空格切分，不适合没有空格的中文句子。
下面只增加一个简单 2/3 字符 n-gram tokenizer，然后直接调用 invoke。

它返回 Document，不暴露 BM25 原始分数。需要分数或 RRF 诊断时，应像 level1 那样
直接使用 rank_bm25。

In [6]:
lexical_pattern = re.compile(
    r"[\u3400-\u9fff]+|[a-z0-9]+(?:[._/-][a-z0-9]+)*"
)


def tokenize(text: str) -> list[str]:
    normalized = unicodedata.normalize("NFKC", text).casefold()
    tokens: list[str] = []
    for match in lexical_pattern.finditer(normalized):
        value = match.group(0)
        if "\u3400" <= value[0] <= "\u9fff":
            if len(value) == 1:
                tokens.append(value)
            else:
                tokens.extend(value[i : i + 2] for i in range(len(value) - 1))
                tokens.extend(value[i : i + 3] for i in range(len(value) - 2))
        else:
            tokens.append(value)
    return tokens


bm25_retriever = BM25Retriever.from_documents(
    chunks,
    preprocess_func=tokenize,
    k=4,
)
bm25_docs = bm25_retriever.invoke(query)
print(
    [
        {
            "source": doc.metadata["source_name"],
            "chunk_id": doc.metadata["chunk_id"],
        }
        for doc in bm25_docs
    ]
)

[{'source': '110kV黄金输变电工程三级进度计划.txt', 'chunk_id': 'doc-b94ad48bc283:chunk-000'}, {'source': '110kV黄金输变电工程三级进度计划.txt', 'chunk_id': 'doc-b94ad48bc283:chunk-001'}, {'source': '珠海110kV江湾输变电工程总体进度计划横道图.txt', 'chunk_id': 'doc-00b4edd811b2:chunk-000'}, {'source': '110千伏节点计划-重点关注.txt', 'chunk_id': 'doc-eb38ed011147:chunk-011'}]


## 6. Retriever Tool：很薄的适配层

create_retriever_tool 只是把已经能工作的 BaseRetriever 包装成结构化 Tool。
它不会改进 chunk、embedding 或排序，也不会自动完成 Evidence Gate。

这里直接调用 Tool 验证接口，不启动 Agent、不调用 LLM。

In [7]:
retrieval_tool = create_retriever_tool(
    similarity_retriever,
    name="search_gold_project_chunks",
    description="检索黄金输变电工程文档中的相关文本片段。",
)
tool_output = retrieval_tool.invoke({"query": query})
print(
    {
        "tool_name": retrieval_tool.name,
        "output_chars": len(tool_output),
        "preview": " ".join(tool_output.split())[:180],
    }
)

{'tool_name': 'search_gold_project_chunks', 'output_chars': 1989, 'preview': '以下内容来自PDF第1页。 该进度计划的完整名称为珠海110千伏黄金输变电工程工程进度计划横道图。 标识号1是顶层独立任务“施工合同签定”。计划开始2024年4月25日，计划完成2024年4月30日。 标识号2是顶层独立任务“施工准备”。计划开始2024年5月1日，计划完成2024年5月23日。 标识号3是顶层独立任务“开工报审”。计划开始2024年5月24'}


## 7. 其他 Retriever 先理解适用条件，不堆实现

| 组件 | 解决的问题 | 当前小知识库是否需要 |
| --- | --- | --- |
| MultiQueryRetriever | 用多种问法扩大召回 | 暂不需要，先扩评测集 |
| SelfQueryRetriever | 让模型把自然语言变成 metadata filter | 不需要，工程别名可确定性处理 |
| ParentDocumentRetriever | 小 chunk 检索、大上下文返回 | 只有真实出现记录被切断时再引入 |
| MultiVectorRetriever | 一个父文档对应多种检索表示 | 当前 8 文档没有必要 |
| ContextualCompressionRetriever | 对候选再过滤/压缩 | 需要额外模型与成本，当前不启用 |
| Ensemble / RRF | 融合 Dense 与 Sparse 排名 | level1 已用最小代码演示 |

这些组件位于 retrieval strategy / orchestration 层。它们不替代 embedding、向量索引，
也不保证答案事实正确。

## 8. 结论

- VectorStore 负责存储和搜索；with_score 在这一层最直接；
- Retriever 把搜索统一成 Runnable，便于 invoke/batch 和后续链式组合；
- Tool 只是 Retriever 的调用适配层；
- similarity、MMR、BM25 解决不同召回问题，不等于证据校验；
- 当前小型知识库使用 InMemoryVectorStore 已足够，无需持久化 Chroma。